# compare_all_methods - all approaches at a matched budget

Self-contained. For each seed, `fixed_FPR` (POUNDERS) sets the matched revealed-shot budget, then
every other method runs at that same budget:

| folder | method |
|---|---|
| `fixed_fpr/`    | fixed_FPR (POUNDERS, uniform on FPR circuits) - budget anchor |
| `fixed_no_fpr/` | no_FPR (POUNDERS, uniform all circuits) |
| `adaptive_D/`   | adaptive_FPR (POUNDERS, D-optimal) |
| `adaptive_A/`   | adaptive_FPR (POUNDERS, A-optimal) |
| `adaptive_L/`   | adaptive_FPR (POUNDERS, L-optimal) |
| `lm/`           | pyGSTi built-in LM solver (uniform, matched budget) |

Writes `all_methods_summary.csv`. **Needs PyROL -> run in Docker.** Heavy: adaptive-L is slow (gauge-opt
metric) and LM is a separate fit, so start with **one seed**.

In [ ]:
import sys
from pathlib import Path
candidates = [
    Path.cwd(), Path.cwd() / "seed_sweep_experiments",
    Path.cwd() / "GST_POUNDERS" / "seed_sweep_experiments",
    Path("/workspace/IBCDFO/GST_POUNDERS/seed_sweep_experiments"),
]
EXPERIMENT_DIR = next((p.resolve() for p in candidates if (p / "gst_seed_experiment.py").exists()), None)
if EXPERIMENT_DIR is None:
    raise FileNotFoundError("Could not locate gst_seed_experiment.py")
if str(EXPERIMENT_DIR) not in sys.path:
    sys.path.insert(0, str(EXPERIMENT_DIR))
from gst_seed_experiment import ExperimentConfig, run_one_experiment, GSTProblem
config = ExperimentConfig.from_json(EXPERIMENT_DIR / "experiment_config.json")
print("Experiment dir:", EXPERIMENT_DIR, "| noise_model:", config.noise_model, "| model_kind:", config.model_kind)

In [ ]:
# ------------------------- knobs -------------------------
SEEDS       = "101:121"                          # start with 1 seed; e.g. "101:106" for 101-105
FIXED_FPR_SHOTS = 800   # shots/circuit for the anchor; fixed_FPR's accounted cost = budget for all others
RESULTS_DIR = EXPERIMENT_DIR / "all_methods_comparison"
FORCE       = False
ADAPTIVE_ONLY = False   # True: load fixed_FPR/no_FPR/LM from disk, re-run ONLY adaptive D/A/L
LM_MODES, LM_MAXITER = "CPTPLND", 800            # pyGSTi LM baseline settings

In [ ]:
import json
from dataclasses import replace, asdict
import numpy as np
import pandas as pd

INF = "mean_gate_entanglement_infidelity_to_truth"

def parse_seeds(spec):
    seeds = []
    for tok in str(spec).split(","):
        tok = tok.strip()
        if not tok: continue
        if ":" not in tok: seeds.append(int(tok)); continue
        parts = [int(v) for v in tok.split(":")]
        start, stop = parts[0], parts[1]; step = parts[2] if len(parts) == 3 else 1
        seeds.extend(range(start, stop, step))
    return sorted(set(seeds))

def _completed(rd): return (rd / "completed.json").exists() and (rd / "summary.json").exists()
def _config_matches(rd, cfg):
    p = rd / "config.json"
    return p.exists() and json.loads(p.read_text()) == json.loads(json.dumps(asdict(cfg)))

def run_or_load(cfg, seed, method, rd, force):
    """POUNDERS methods via run_one_experiment (with skip-if-done)."""
    if _completed(rd) and not force and _config_matches(rd, cfg):
        print(f"SKIP seed={seed} {rd.name}: completed"); return json.loads((rd / "summary.json").read_text())
    print(f"RUN  seed={seed} {rd.name}")
    return run_one_experiment(config=cfg, data_seed=seed, method=method, output_dir=rd)

def fit_or_load_lm(cfg, seed, target_shots, lm_dir, force):
    """pyGSTi LM on uniform shots at the matched budget; writes summary.json + lm_trajectory.csv."""
    if (lm_dir / "summary.json").exists() and (lm_dir / "lm_trajectory.csv").exists() and not force:
        print(f"SKIP seed={seed} lm: completed"); return json.loads((lm_dir / "summary.json").read_text())
    print(f"RUN  seed={seed} lm")
    import re, pygsti
    from pygsti.optimize import SimplerLMOptimizer
    SPAM = "mean_spam_vector_l2_error_to_truth"
    prob = GSTProblem(cfg, seed)
    try: prob.base_model.sim = "map"; prob.truth_model.sim = "map"
    except Exception: pass
    per = max(1, round(target_shots / len(prob.circuits)))
    shots = prob.normalize_shots(per)
    dataset = prob.simulate_dataset(shots, seed=9_000_000 + seed)
    data = pygsti.protocols.ProtocolData(prob.design, dataset)
    opt = SimplerLMOptimizer(maxiter=LM_MAXITER, maxfev=LM_MAXITER, tol=1e-6, init_munu="auto", oob_action="reject")
    proto = pygsti.protocols.StandardGST(modes=LM_MODES, target_model=prob.target_model, optimizer=opt, verbosity=0)
    res = proto.run(data)
    keys = list(res.estimates.keys()); est_key = LM_MODES if LM_MODES in res.estimates else keys[0]
    est = res.estimates[est_key]
    fit = est.models["final iteration estimate"]
    summ, _, _ = prob.aligned_error_metrics(fit, prob.truth_model, "truth")

    # per-max-length-stage convergence trajectory (one GST estimate per max-length)
    mls = list(getattr(cfg, "max_lengths", []))
    stage_keys = sorted([k for k in est.models if re.fullmatch(r"iteration \d+ estimate", k)],
                        key=lambda k: int(k.split()[1]))
    traj = []
    for si, k in enumerate(stage_keys):
        try:
            st, _, _ = prob.aligned_error_metrics(est.models[k], prob.truth_model, "truth")
            traj.append({"stage": si, "max_length": (mls[si] if si < len(mls) else si),
                         INF: float(st[INF]), SPAM: float(st.get(SPAM, float("nan")))})
        except Exception:
            pass

    out = {"data_seed": seed, "method": "lm",
           INF: float(summ[INF]),
           "accounted_revealed_shots": int(shots.sum()), "physical_precomputed_shots": int(shots.sum()),
           "max_shots_per_circuit": int(shots.max()), "min_shots_per_circuit": int(shots.min()),
           "mean_shots_per_circuit": float(shots.mean()),
           "total_circuits": int(len(prob.circuits)), "revealed_circuits": int(len(prob.circuits)),
           "flag": "LM"}
    lm_dir.mkdir(parents=True, exist_ok=True)
    (lm_dir / "summary.json").write_text(json.dumps(out, indent=2))
    pd.DataFrame(traj).to_csv(lm_dir / "lm_trajectory.csv", index=False)
    return out

def _row(label, seed, s):
    return {"seed": seed, "method": label, INF: s.get(INF, float("nan")),
            "accounted_revealed_shots": s.get("accounted_revealed_shots"),
            "max_shots_per_circuit": s.get("max_shots_per_circuit"), "flag": s.get("flag")}

In [ ]:
# ---- run all 6 methods per seed; fixed_FPR is the budget anchor ----
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
rows = []
for seed in parse_seeds(SEEDS):
    seed_dir = RESULTS_DIR / f"seed_{seed:06d}"; seed_dir.mkdir(parents=True, exist_ok=True)

    # fixed_FPR runs first at FIXED_FPR_SHOTS/circuit; its accounted cost = the matched budget.
    if ADAPTIVE_ONLY and (seed_dir / "fixed_fpr" / "summary.json").exists():
        fixed = json.loads((seed_dir / "fixed_fpr" / "summary.json").read_text())
        print(f"  LOAD seed={seed} fixed_fpr (adaptive-only)")
    else:
        fixed = run_or_load(replace(config, fixed_fpr_shots=int(FIXED_FPR_SHOTS)), seed, "fixed_fpr",
                            seed_dir / "fixed_fpr", FORCE)
    target = int(fixed["accounted_revealed_shots"]); total_circuits = int(fixed["total_circuits"])
    rows.append(_row("fixed_FPR", seed, fixed))
    print(f"  seed={seed}: budget anchor (fixed_FPR accounted) = {target:,}")

    # no_FPR: uniform over ALL circuits at target // total_circuits -> total ~ target
    no_fpr_shots = max(1, target // total_circuits)
    if ADAPTIVE_ONLY and (seed_dir / "fixed_no_fpr" / "summary.json").exists():
        nf = json.loads((seed_dir / "fixed_no_fpr" / "summary.json").read_text())
        print(f"  LOAD seed={seed} no_FPR (adaptive-only)")
    else:
        nf = run_or_load(replace(config, fixed_no_fpr_shots=no_fpr_shots), seed, "fixed_no_fpr",
                         seed_dir / "fixed_no_fpr", FORCE)
    rows.append(_row("no_FPR", seed, nf))

    # adaptive D/A/L: acquire up to the anchor budget (lands a few % over -- baseline drag)
    for crit in ["D", "A", "L"]:
        acfg = replace(config, adaptive_criterion=crit, adaptive_total_shot_budget=target)
        try:
            res = run_or_load(acfg, seed, "adaptive_fpr", seed_dir / f"adaptive_{crit}", FORCE)
            rows.append(_row(f"adaptive_{crit}", seed, res))
        except Exception as exc:
            print(f"  seed={seed} adaptive_{crit}: FAILED {exc!r}")
            rows.append({"seed": seed, "method": f"adaptive_{crit}", INF: float("nan"),
                         "accounted_revealed_shots": 0, "max_shots_per_circuit": 0, "flag": "FAILED"})

    # LM: uniform over all circuits at the anchor budget
    if ADAPTIVE_ONLY and (seed_dir / "lm" / "summary.json").exists():
        lm = json.loads((seed_dir / "lm" / "summary.json").read_text())
        print(f"  LOAD seed={seed} LM (adaptive-only)")
        rows.append(_row("LM", seed, lm))
    else:
        try:
            lm = fit_or_load_lm(config, seed, target, seed_dir / "lm", FORCE)
            rows.append(_row("LM", seed, lm))
        except Exception as exc:
            print(f"  seed={seed} LM: FAILED {exc!r}")
            rows.append({"seed": seed, "method": "LM", INF: float("nan"),
                         "accounted_revealed_shots": 0, "max_shots_per_circuit": 0, "flag": "FAILED"})

    pd.DataFrame(rows).to_csv(RESULTS_DIR / "all_methods_summary.csv", index=False)
    print(f"seed {seed} done -> budget ~{target:,}")

df = pd.DataFrame(rows)
df.to_csv(RESULTS_DIR / "all_methods_summary.csv", index=False)
print("\nsaved", RESULTS_DIR / "all_methods_summary.csv")

In [ ]:
# ---- quick table ----
print(df[["seed", "method", INF, "accounted_revealed_shots", "max_shots_per_circuit", "flag"]].to_string(index=False))
print("\n=== median infidelity by method ===")
print(df.groupby("method")[INF].median().sort_values().to_string())

Now open **`analyze_all_methods.ipynb`** for the comparison plots (grouped bars + shot efficiency).